In [0]:
create schema if not exists f1.pipeline

In [0]:
SELECT driver_id, points, grand_prix_wins, sprint_wins, disqualifications
FROM f1.pipeline.gold_season_points
WHERE year = 2025
ORDER BY points DESC
LIMIT 3

In [0]:
SELECT q.timestamp,
       q.e.name           AS rule,
       q.e.passed_records AS passed,
       q.e.failed_records AS failed
FROM (
  SELECT timestamp,
         explode(from_json(
           details:flow_progress:data_quality:expectations,
           'array<struct<name:string, dataset:string, passed_records:bigint, failed_records:bigint>>'
         )) AS e
  FROM event_log(TABLE(f1.pipeline.silver_session_results))
  WHERE event_type = 'flow_progress'
    AND details:flow_progress:data_quality:expectations IS NOT NULL
) q
ORDER BY q.timestamp DESC, rule

In [0]:
SELECT 'meetings'       AS t, (SELECT COUNT(*) FROM f1.bronze.openf1_meetings)       AS notebook, (SELECT COUNT(*) FROM f1.pipeline.bronze_meetings)       AS pipeline
UNION ALL SELECT 'sessions',       (SELECT COUNT(*) FROM f1.bronze.openf1_sessions),       (SELECT COUNT(*) FROM f1.pipeline.bronze_sessions)
UNION ALL SELECT 'drivers',        (SELECT COUNT(*) FROM f1.bronze.openf1_drivers),        (SELECT COUNT(*) FROM f1.pipeline.bronze_drivers)
UNION ALL SELECT 'session_result', (SELECT COUNT(*) FROM f1.bronze.openf1_session_result), (SELECT COUNT(*) FROM f1.pipeline.bronze_session_result)
UNION ALL SELECT 'starting_grid',  (SELECT COUNT(*) FROM f1.bronze.openf1_starting_grid),  (SELECT COUNT(*) FROM f1.pipeline.bronze_starting_grid)
UNION ALL SELECT 'laps',           (SELECT COUNT(*) FROM f1.bronze.openf1_laps),           (SELECT COUNT(*) FROM f1.pipeline.bronze_laps)
UNION ALL SELECT 'stints',         (SELECT COUNT(*) FROM f1.bronze.openf1_stints),         (SELECT COUNT(*) FROM f1.pipeline.bronze_stints)
UNION ALL SELECT 'pit',            (SELECT COUNT(*) FROM f1.bronze.openf1_pit),            (SELECT COUNT(*) FROM f1.pipeline.bronze_pit)
UNION ALL SELECT 'weather',        (SELECT COUNT(*) FROM f1.bronze.openf1_weather),        (SELECT COUNT(*) FROM f1.pipeline.bronze_weather)
UNION ALL SELECT 'race_control',   (SELECT COUNT(*) FROM f1.bronze.openf1_race_control),   (SELECT COUNT(*) FROM f1.pipeline.bronze_race_control)

In [0]:
SELECT 'sessions'          AS t, (SELECT COUNT(*) FROM f1.silver.sessions)          AS notebook, (SELECT COUNT(*) FROM f1.pipeline.silver_sessions)          AS pipeline
UNION ALL SELECT 'meetings',          (SELECT COUNT(*) FROM f1.silver.meetings),          (SELECT COUNT(*) FROM f1.pipeline.silver_meetings)
UNION ALL SELECT 'session_results',   (SELECT COUNT(*) FROM f1.silver.session_results),   (SELECT COUNT(*) FROM f1.pipeline.silver_session_results)
UNION ALL SELECT 'session_drivers',   (SELECT COUNT(*) FROM f1.silver.session_drivers),   (SELECT COUNT(*) FROM f1.pipeline.silver_session_drivers)
UNION ALL SELECT 'starting_grid',     (SELECT COUNT(*) FROM f1.silver.starting_grid),     (SELECT COUNT(*) FROM f1.pipeline.silver_starting_grid)
UNION ALL SELECT 'laps',              (SELECT COUNT(*) FROM f1.silver.laps),              (SELECT COUNT(*) FROM f1.pipeline.silver_laps)
UNION ALL SELECT 'stints',            (SELECT COUNT(*) FROM f1.silver.stints),            (SELECT COUNT(*) FROM f1.pipeline.silver_stints)
UNION ALL SELECT 'quarantine_stints', (SELECT COUNT(*) FROM f1.silver.quarantine_stints), (SELECT COUNT(*) FROM f1.pipeline.silver_quarantine_stints)
UNION ALL SELECT 'pit_stops',         (SELECT COUNT(*) FROM f1.silver.pit_stops),         (SELECT COUNT(*) FROM f1.pipeline.silver_pit_stops)
UNION ALL SELECT 'weather',           (SELECT COUNT(*) FROM f1.silver.weather),           (SELECT COUNT(*) FROM f1.pipeline.silver_weather)
UNION ALL SELECT 'race_control',      (SELECT COUNT(*) FROM f1.silver.race_control),      (SELECT COUNT(*) FROM f1.pipeline.silver_race_control)

In [0]:
SELECT 'dim_driver'            AS t, (SELECT COUNT(*) FROM f1.gold.dim_driver)                AS notebook, (SELECT COUNT(*) FROM f1.pipeline.gold_dim_driver)                AS pipeline
UNION ALL SELECT 'dim_session',           (SELECT COUNT(*) FROM f1.gold.dim_session),               (SELECT COUNT(*) FROM f1.pipeline.gold_dim_session)
UNION ALL SELECT 'fact_session_result',   (SELECT COUNT(*) FROM f1.gold.fact_session_result),       (SELECT COUNT(*) FROM f1.pipeline.gold_fact_session_result)
UNION ALL SELECT 'fact_lap',              (SELECT COUNT(*) FROM f1.gold.fact_lap),                  (SELECT COUNT(*) FROM f1.pipeline.gold_fact_lap)
UNION ALL SELECT 'agg_tyre_degradation',  (SELECT COUNT(*) FROM f1.gold.agg_tyre_degradation),      (SELECT COUNT(*) FROM f1.pipeline.gold_agg_tyre_degradation)
UNION ALL SELECT 'agg_championship',      (SELECT COUNT(*) FROM f1.gold.agg_championship_progress), (SELECT COUNT(*) FROM f1.pipeline.gold_agg_championship_progress)

In [0]:
SELECT driver_id, total_points
FROM f1.pipeline.gold_agg_championship_progress
WHERE year = 2025 AND round_number = 24
ORDER BY total_points DESC
LIMIT 3

In [0]:
SELECT COUNT_IF(compound IS NULL) AS laps_without_tyre
FROM f1.pipeline.gold_fact_lap

In [0]:
SELECT q.timestamp,
       q.e.dataset        AS table_name,
       q.e.name           AS rule,
       q.e.passed_records AS passed,
       q.e.failed_records AS failed
FROM (
  SELECT timestamp,
         explode(from_json(
           details:flow_progress:data_quality:expectations,
           'array<struct<name:string, dataset:string, passed_records:bigint, failed_records:bigint>>'
         )) AS e
  FROM event_log(TABLE(f1.pipeline.gold_fact_lap))
  WHERE event_type = 'flow_progress'
    AND details:flow_progress:data_quality:expectations IS NOT NULL
) q
WHERE q.e.name IN ('has_tyre', 'has_driver_key')
ORDER BY q.timestamp DESC
LIMIT 10

In [0]:
DROP TABLE IF EXISTS f1.gold.dim_driver;
DROP TABLE IF EXISTS f1.gold.dim_session;
DROP TABLE IF EXISTS f1.gold.fact_session_result;
DROP TABLE IF EXISTS f1.gold.fact_lap;
DROP TABLE IF EXISTS f1.gold.agg_tyre_degradation;
DROP TABLE IF EXISTS f1.gold.agg_championship_progress;

DROP TABLE IF EXISTS f1.silver.sessions;
DROP TABLE IF EXISTS f1.silver.meetings;
DROP TABLE IF EXISTS f1.silver.session_results;
DROP TABLE IF EXISTS f1.silver.session_drivers;
DROP TABLE IF EXISTS f1.silver.starting_grid;
DROP TABLE IF EXISTS f1.silver.laps;
DROP TABLE IF EXISTS f1.silver.quarantine_laps;
DROP TABLE IF EXISTS f1.silver.stints;
DROP TABLE IF EXISTS f1.silver.quarantine_stints;
DROP TABLE IF EXISTS f1.silver.pit_stops;
DROP TABLE IF EXISTS f1.silver.weather;
DROP TABLE IF EXISTS f1.silver.race_control;

DROP TABLE IF EXISTS f1.bronze.openf1_meetings;
DROP TABLE IF EXISTS f1.bronze.openf1_sessions;
DROP TABLE IF EXISTS f1.bronze.openf1_drivers;
DROP TABLE IF EXISTS f1.bronze.openf1_session_result;
DROP TABLE IF EXISTS f1.bronze.openf1_starting_grid;
DROP TABLE IF EXISTS f1.bronze.openf1_laps;
DROP TABLE IF EXISTS f1.bronze.openf1_stints;
DROP TABLE IF EXISTS f1.bronze.openf1_pit;
DROP TABLE IF EXISTS f1.bronze.openf1_weather;
DROP TABLE IF EXISTS f1.bronze.openf1_race_control;

DROP VOLUME IF EXISTS f1.bronze.checkpoints;   -- Auto Loader's memory, no longer used